In [1]:
import pandas as pd

In [2]:

excel_file = pd.ExcelFile(
    r"C:\Users\HP\Documents\Data Analysis\Python Analytics\Airtel RAW 8th Aug'26.xlsx"
)

In [3]:
airtel_df = pd.read_excel(excel_file, sheet_name=0)

In [4]:
airtel_df = airtel_df.rename(columns = {
    "Unnamed: 0" : "Interval",
    "Unnamed: 1" : "Queue/SKillName",
    "Unnamed: 2" : "Calls Offered",
    "Unnamed: 3" : "Calls Answered",
    "Unnamed: 4" : "Calls Abandoned",
    "Unnamed: 5" : "ACWTime",
    "Unnamed: 6" : "Talk Time",
    "Unnamed: 7" : "Calls Answered within Threshold",
    "Unnamed: 8" : "Calls Answered After Threshold",
    "Unnamed: 9" : "Calls Abandoned Within Threshold",
    "Unnamed: 10" : "Calls Abandoned After Threshold",
    "Unnamed: 11" : "Wait Time",
    "Unnamed: 12" : "Max Wait Time",
    "Unnamed: 13" : "HoldTime",
    "Unnamed: 14" : "CallsAbandToAgent",
    "Unnamed: 15" : "CallsAbandQ",
    "Unnamed: 16" : "CallsHandled",
    "Unnamed: 17" : "CallsAbandDequeued",
    "Unnamed: 18" : "RONA",
    "Unnamed: 19" : "RouterError"
        }
    )

In [5]:
airtel_df = airtel_df.drop([0, 1]).reset_index(drop=True)

In [6]:
def remove_buttom_row(df):
    return df.iloc[:-3].reset_index(drop=True)

In [7]:
airtel_df = remove_buttom_row(airtel_df)

In [8]:
# ACCESSING THE HOURLY AGENT EFFICIENCY RAW

In [9]:
agent_eff_raw = pd.read_excel(excel_file, sheet_name=1)

In [10]:
agent_eff_raw = agent_eff_raw.rename(columns = {
    "Offered Parameters" : "CallsAbandToAgent",
    "Unnamed: 0" : "Agent Name/ID",
    "Unnamed: 1" : "Interval",
    "Unnamed: 2" : "Connection Time",
    "Unnamed: 3" : "Calls Presented",
    "Unnamed: 4" : "Calls Accepted",
    "Unnamed: 5" : "Talk Time",
    "Unnamed: 6" : "Waiting Time",
    "Unnamed: 7" : "Idle Time",
    "Unnamed: 8" : "Not Ready",
    "Unnamed: 9" : "Hold Time",
    "Unnamed: 10" : "AHT",
    
        }
    )

In [11]:
agent_eff_raw = agent_eff_raw.drop(index=0).reset_index(drop=True)

In [12]:
agent_eff_raw = remove_buttom_row(agent_eff_raw)

In [13]:

# Inserting Active+Hold to the table as we will use it for other calculations

agent_eff_raw.insert(
    11,
    "Active+Hold",
    (
        pd.to_timedelta(agent_eff_raw["Talk Time"].astype(str))
        + pd.to_timedelta(agent_eff_raw["Hold Time"].astype(float), unit="s")
        + pd.to_timedelta(agent_eff_raw["Idle Time"].astype(str))
    ).dt.total_seconds() / 86400
)

# agent_eff_raw["Active+Hold"] = agent_eff_raw["Active+Hold"].round(2)

In [14]:
# Convert Columns to timedelta so we can get decimals since they are in time format


agent_eff_raw["Connection Time"] = pd.to_timedelta(agent_eff_raw["Connection Time"])
agent_eff_raw["Not Ready"] = pd.to_timedelta(agent_eff_raw["Not Ready"])

# Then insert the Login column
agent_eff_raw.insert(
    12,
    "Login",
    (
        agent_eff_raw["Connection Time"].dt.total_seconds() / 86400
        + agent_eff_raw["Not Ready"].dt.total_seconds() / 86400
    )
)

In [15]:
# Inserting a column for idle time that already exist, but we are converting it to seconds for proper calculations
agent_eff_raw.insert(
    13,
    "Idle_Time",
    pd.to_timedelta(
        agent_eff_raw["Idle Time"].astype(str)
    ).dt.total_seconds() / 86400
)

In [16]:
# Inserting a column for NOT READY that already exist, but we are converting it to seconds for proper calculations
agent_eff_raw.insert(
    14,
    "Not_Ready",
    pd.to_timedelta(
        agent_eff_raw["Not Ready"].astype(str)
    ).dt.total_seconds() / 86400
)

In [17]:
agent_eff_raw["Interval"] = pd.to_datetime(
    agent_eff_raw["Interval"],
    format="%m/%d/%y %I:%M:%S %p"
)


In [18]:
#DONE WITH CLEANING. ANALYSIS CODES BELOW

In [19]:
# QUEUE ANALYSIS

In [20]:
# Convert the Interval column to datetime
airtel_df["Interval"] = pd.to_datetime(
    airtel_df["Interval"],
    format="%m/%d/%y %I:%M:%S %p"
)


In [21]:
# Create half-hour report from the raw
# What goes here are what we have available in the RAW data imported
half_hourly_df = (
    airtel_df
    .groupby("Interval", as_index=False)[[
        "Calls Answered",
        "Calls Abandoned",
        "Calls Answered within Threshold",
        "Calls Answered After Threshold",
        "Calls Abandoned Within Threshold",
        "Talk Time",
        "HoldTime",
        "ACWTime",
        "CallsHandled",
    ]]
    .sum()
    .sort_values("Interval")
)

In [22]:

#Data to be added to report after creating data frames
#We are not busing the offered from the df because it gives negative figures sometimes while calculating Calls Abandoned

half_hourly_df.insert(
    1,
    "Calls Offered",
    half_hourly_df["Calls Answered"] +
    half_hourly_df["Calls Abandoned"]
)

In [23]:

#Data to be added to report after creating data frames

half_hourly_df.insert(1, "Forecast", [118, 82,	53,	38,	37,	28,	22,	26,	28,	38,	46,	91,	195,345,478,578,651,676,723,703,687,654,646,714,
                                      609,598,598,616,623,656,662,676,604,642,622,597,613,676,810,845, 967,937,808,711,553,412,312,194
                                     ])


In [24]:
# Remove decimals for half hourly
#This Cell now carries the New Headers

half_hourly_copy = half_hourly_df.copy()

half_hourly_copy[[
    "Calls Offered",
    "Calls Answered",
    "Calls Answered within Threshold",
    "Calls Answered After Threshold",
    "Calls Abandoned Within Threshold",
    "Talk Time",
    "HoldTime",
    "ACWTime",
    "CallsHandled",
    "Forecast"
]] = half_hourly_copy[[
    "Calls Offered",
    "Calls Answered",
    "Calls Answered within Threshold",
    "Calls Answered After Threshold",
    "Calls Abandoned Within Threshold",
    "Talk Time",
    "HoldTime",
    "ACWTime",
    "CallsHandled",
    "Forecast"
]].astype(int)

In [25]:
# Calculate Calls Abandoned
half_hourly_copy["Calls Abandoned"] = ( 
    half_hourly_copy["Calls Offered"] - 
    half_hourly_copy["Calls Answered"]
)

In [26]:
#Calculate AHT (Average Handeling Time)
# AHT (seconds)
half_hourly_copy["AHT (Seconds)"] = (
    half_hourly_copy["Talk Time"] +
    half_hourly_copy["HoldTime"] +
    half_hourly_copy["ACWTime"]
) / half_hourly_copy["CallsHandled"]

In [27]:
#Calculating FD
half_hourly_copy["FD %"] = (
    half_hourly_copy["Calls Offered"] / 
    half_hourly_copy["Forecast"]
).map("{:.0%}".format)

In [28]:
#Calculating Answered Level
#KPI for the %age answred compared to the offered

half_hourly_copy["AL%"] =(
    half_hourly_copy["Calls Answered"] / 
    half_hourly_copy["Calls Offered"]
).map("{:.0%}".format)

In [29]:
#Calculating Service Level (SL) Level

half_hourly_copy["SL%"] =(
    half_hourly_copy["Calls Answered within Threshold"] / 
    (half_hourly_copy["Calls Offered"] - half_hourly_copy["Calls Abandoned Within Threshold"])
).map("{:.0%}".format)

In [30]:
# Calculate Answered to Forecast
half_hourly_copy ["A2F%"] = (
    half_hourly_copy["Calls Answered"]/
    half_hourly_copy["Forecast"]
).map("{:.0%}".format)


In [31]:
# Calcalte Login Required
half_hourly_copy ["Login Required"] = (
    half_hourly_copy["Calls Offered"]/(
    (30*0.85) / (115/60)  
    )
).round().astype(int)
#30*0.85 is stating that at every interval agent is supposed to be active for 85% of the time
# 115/60 is coverting AHT to seconds. Expected AHT for every interval is 115 seconds

In [32]:
# Adding Login Count to the report
login_sum = agent_eff_raw.groupby("Interval")["Login"].sum()

half_hourly_copy["Login Count"] = (
    half_hourly_copy["Interval"]
    .map(login_sum)
    * 48
).round().astype(int)



In [33]:
# Adding FTE to the report

fte = agent_eff_raw.groupby("Interval")["Active+Hold"].sum()

half_hourly_copy["FTE"] = (
    half_hourly_copy["Interval"]
    .map(fte)
    * 48
).round().astype(int)

In [34]:
# Calculate Deficit/Excess
half_hourly_copy["Deficit/Eccess"] = ( 
    half_hourly_copy["FTE"] - 
    half_hourly_copy["Login Required"]
)

In [35]:
# Adding Not Ready to the report
not_ready = agent_eff_raw.groupby("Interval")["Not_Ready"].sum()

half_hourly_copy["Not_Ready"] = (
    half_hourly_copy["Interval"]
    .map(not_ready)
    * 48
).round().astype(int)

In [36]:
# Adding Idle to the report
idle = agent_eff_raw.groupby("Interval")["Idle_Time"].sum()

half_hourly_copy["Idle_Time"] = (
    half_hourly_copy["Interval"]
    .map(idle)
    * 48
).round().astype(int)

In [37]:
# Calculate Utilization
half_hourly_copy ["Utilization"] = (
    (
        half_hourly_copy["Calls Answered"]*
        half_hourly_copy["AHT (Seconds)"]
    )/
    (
        half_hourly_copy["FTE"] * 
        1800
    )
).map("{:.0%}".format)

In [38]:
# Format the Interval column to show only the time without date
#dec_rem["Interval"] = (
  #  dec_rem["Interval"]
    #.dt.strftime("%I:%M %p")
    #.str.lstrip("0")
#)

In [39]:
#Creating Report
# All columns needed for calculations will be here but i do not want some to be in the report
# That Means all columns MUST be here
report_df = half_hourly_copy[[
    "Interval",
    "Forecast",
    "Calls Offered",
    "Calls Answered",
    "Calls Answered within Threshold",
   "Calls Answered After Threshold",
    "Calls Abandoned Within Threshold",
    "Calls Abandoned",
    "AHT (Seconds)",
    "FD %",
    "AL%",
    "SL%",
    "A2F%",
    "Login Required",
    "Login Count",
    "FTE",
    "Deficit/Eccess",
    "Not_Ready",
    "Idle_Time",
    "Utilization"
]].copy()
report_df["AHT (Seconds)"] = (
    report_df["AHT (Seconds)"]
    .round()
    .astype(int)
)

In [40]:
#Summation for total ROW for %age

#FD Summation for total ROW for %age
total_fd = (
    report_df["Calls Offered"].sum() /
    report_df["Forecast"].sum()
)

total_fd = "{:.0%}".format(total_fd)


#AL Summation for total ROW for %age
total_al = (
    report_df["Calls Answered"].sum() /
    report_df["Calls Offered"].sum()
)

total_al = "{:.0%}".format(total_al)


#SL Summation for total ROW for %age
total_sl = (
    report_df["Calls Answered within Threshold"].sum()
    /
    (
        report_df["Calls Offered"].sum()
        - report_df["Calls Abandoned Within Threshold"].sum()
    )
)

total_sl = "{:.0%}".format(total_sl)

#AHT Summation for total ROW for %age
total_aht = (
    (
        report_df["AHT (Seconds)"].iloc[:-1] *
        report_df["Calls Answered"].iloc[:-1]
    ).sum()
    /
    report_df["Calls Answered"].iloc[:-1].sum()
)

#A2F Summation for total ROW for %age
total_a2f = (
    report_df["Calls Answered"].sum()/
    report_df["Forecast"].sum()
)
total_a2f ="{:.0%}".format(total_a2f)

#Login Required Summation for total ROW
total_log_req = round(report_df["Login Required"].sum() * 30 / 480)

#Login Required Summation for total ROW
total_login_count = round(report_df["Login Count"].sum() * 30 / 480)

#FTE Summation for total ROW
total_fte = round(report_df["FTE"].sum() * 30 / 480)

#Deficit/Eccess Summation for total ROW
overall_fte = report_df["FTE"].sum()
overall_login_required = report_df["Login Required"].sum()
total_deficit = overall_fte - overall_login_required

print(total_deficit)

#Not Ready Summation for total ROW
total_not_ready = round(report_df["Not_Ready"].sum() * 30 / 480)

#Idle Summation for total ROW
total_idle = round(report_df["Idle_Time"].sum() * 30 / 480)

#Utilization Summation for total ROW for %age
total_uttilization = (
    (
        report_df["Calls Answered"]*
        report_df["AHT (Seconds)"]
    ).sum()/
    (
        report_df["FTE"].sum()*
        1800
    )
)
total_uttilization ="{:.0%}".format(total_uttilization)


145


In [41]:
# Building the final report Headers 
#ONLY column to be Added to the report should be here as Headers are updated here
report_df = report_df[
    [
        "Interval",
        "Forecast",
        "Calls Offered",
        "Calls Answered",
        "Calls Abandoned",
        "AHT (Seconds)",
        "FD %",
        "AL%",
        "SL%",
        "A2F%",
        "Login Required",
        "Login Count",
        "FTE",
        "Deficit/Eccess",
        "Not_Ready",
        "Idle_Time",
        "Utilization"
    ]
].copy()

In [42]:
#This code is to stop multiple total reproduction

report_df = report_df[report_df["Interval"] != "Overall"].copy()

In [43]:
# Add Total row, what should appear in the total row will be here
total_row = pd.DataFrame({
    "Interval": ["Overall"],
    "Forecast": [report_df["Forecast"].sum()],
    "Calls Offered": [report_df["Calls Offered"].sum()],
    "Calls Answered": [report_df["Calls Answered"].sum()],
    #"Calls Answered within Threshold" : [report_df["Calls Answered within Threshold"].sum()],
    #"Calls Answered After Threshold" : [report_df["Calls Answered After Threshold"].sum()],
    "Calls Abandoned": [report_df["Calls Abandoned"].sum()],
    "FD %": [total_fd],
    "AHT (Seconds)": [total_aht],   # I'll calculate this later
    "AL%" : [total_al],
    "SL%" : [total_sl],
    "A2F%" : [total_a2f],
    "Login Required": [total_log_req],
    "Login Count": [total_login_count],
    "FTE": [total_fte],
    "Deficit/Eccess": [total_deficit],
    "Not_Ready": [total_not_ready],
    "Idle_Time": [total_idle],
    "Utilization": [total_uttilization]
})
#Total row 
report_df = pd.concat([report_df, total_row], ignore_index=True)

# Convert whole-number columns back to integers (Removing Decimals fom the report)
int_columns = [
    "AHT (Seconds)",
    "Login Required",
    "Login Count",
    "FTE",
    "Deficit/Eccess",
    "Not_Ready",
    "Idle_Time",
]
# report_df[int_columns] = report_df[int_columns].astype(int)
report_df[int_columns] = report_df[int_columns].round().astype("Int64")

report_df.style.hide(axis="index")


Interval,Forecast,Calls Offered,Calls Answered,Calls Abandoned,AHT (Seconds),FD %,AL%,SL%,A2F%,Login Required,Login Count,FTE,Deficit/Eccess,Not_Ready,Idle_Time,Utilization
2026-08-08 00:00:00,118,138,137,1,114,117%,99%,99%,116%,10,30,26,16,3,18,33%
2026-08-08 00:30:00,82,83,78,5,132,101%,94%,90%,95%,6,13,11,5,3,5,52%
2026-08-08 01:00:00,53,40,36,4,168,75%,90%,90%,68%,3,11,10,7,0,7,34%
2026-08-08 01:30:00,38,40,39,1,134,105%,98%,95%,103%,3,9,8,5,0,5,36%
2026-08-08 02:00:00,37,23,23,0,165,62%,100%,100%,62%,2,8,7,5,0,5,30%
2026-08-08 02:30:00,28,30,30,0,188,107%,100%,97%,107%,2,7,7,5,0,3,45%
2026-08-08 03:00:00,22,28,25,3,185,127%,89%,85%,114%,2,7,6,4,1,4,43%
2026-08-08 03:30:00,26,16,16,0,125,62%,100%,100%,62%,1,7,4,3,2,3,28%
2026-08-08 04:00:00,28,13,13,0,221,46%,100%,92%,46%,1,7,6,5,1,4,27%
2026-08-08 04:30:00,38,28,27,1,147,74%,96%,93%,71%,2,6,6,4,0,4,37%


In [43]:
report_df.to_excel("Airtel_Report.xlsx", index=False)